# Llibreries

In [2]:
import pandas as pd
import numpy as np
import json
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [12]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dimensions.head()

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
0,1,SEXE,1,Dona,Mujer,Female
1,1,SEXE,2,Home,Hombre,Male
2,2,EDAT_1,0,0 anys,0 años,0 years
3,2,EDAT_1,1,1 anys,1 años,1 years
4,2,EDAT_1,2,2 anys,2 años,2 years


In [3]:
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")
dim_barris.head()

,codi_districte,nom_districte,codi_barri,nom_barri,geometria_etrs89,geometria_wgs84
0,1,Ciutat Vella,1,el Raval,"POLYGON ((430162.1875 4581936.9845, 430102.838...","POLYGON ((2.16471378585589 41.3859301967194, 2..."
1,1,Ciutat Vella,2,el Barri Gòtic,"POLYGON ((431189.9075 4581851.4475, 431025.789...","POLYGON ((2.1770141884741 41.385248355328, 2.1..."
2,1,Ciutat Vella,3,la Barceloneta,"POLYGON ((432798.7341255 4582081.2599495, 4327...","POLYGON ((2.19622882469513 41.387454220446, 2...."
3,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera","POLYGON ((431733.736 4582441.816, 431557.5115 ...","POLYGON ((2.18345134701381 41.3906119681235, 2..."
4,2,Eixample,5,el Fort Pienc,"POLYGON ((431741.8152 4582625.6491, 432012.183...","POLYGON ((2.18352725722411 41.3922683849226, 2..."


# Dades Demogràfiques

En aquest apartat enriquirem els datasets amb algunes variables porcentuals que ens ajudaran a modelar segons el nostre objectiu.
- Població Total
- % població estrangera
- % població espanyola
- % estragners_occidentals (Amèrica del Nord i Europa) - Article Lopez Gay o Cocola Gant?
- % pct_universitaris/estudis_superiors
- % pct_sense_estudis


Finalment, eliminarem les columnes innecessaries i exportarem fitxer per a cada any net.

In [13]:
dimensions[dimensions["Desc_Dimensio"] == "NACIONALITAT_REGIO"]["Desc_Valor_CA"].unique()

array(['Àfrica oriental', 'Àfrica central', 'Àfrica septentrional',
       'Àfrica meridional', 'Àfrica occidental', 'Carib',
       'Amèrica central', 'Amèrica del sud', 'Amèrica del nord',
       'Àsia central', 'Àsia oriental', 'Àsia meridional',
       'Àsia sud-oriental', 'Àsia occidental', 'Europa oriental',
       'Europa septentrional', 'Europa meridional', 'Europa occidental',
       'Austràlia i Nova Zelanda', 'Melanèsia', 'Micronèsia', 'Polinèsia',
       'No consta'], dtype=object)

En el càlcul de població estrangera occidental considerarem només les regions d' Amèrica del nord i Europa. Per tant, els valors per al càlcul són:
- Amèrica del nord ()
- Resta de la Unió Europea (Població per barri per nacionalitat)

Aquest indicador ens permetrà estudiar com atribut individual la immigració de països amb major poder adquisitiu.

In [33]:
poblacio_23_raw = pd.read_csv("../data/processed/df_demografia_23.csv")
poblacio_23_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 44 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   codi_barri                     73 non-null     int64  
 1   poblacio_total                 73 non-null     int64  
 2   espanya                        73 non-null     int64  
 3   no_consta_x                    73 non-null     int64  
 4   resta_de_la_uni_europea        73 non-null     int64  
 5   resta_del_mn                   73 non-null     int64  
 6   amrica_central                 73 non-null     int64  
 7   amrica_del_nord                73 non-null     int64  
 8   amrica_del_sud                 73 non-null     int64  
 9   austrlia_i_nova_zelanda        73 non-null     int64  
 10  carib                          73 non-null     int64  
 11  europa_meridional              73 non-null     int64  
 12  europa_occidental              73 non-null     int64

In [35]:
poblacio_15_raw = pd.read_csv("../data/processed/df_demografia_15.csv")
poblacio_15_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 42 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   codi_barri                     73 non-null     int64  
 1   poblacio_total                 73 non-null     int64  
 2   espanya                        73 non-null     int64  
 3   no_consta_x                    73 non-null     int64  
 4   resta_de_la_uni_europea        73 non-null     int64  
 5   resta_del_mn                   73 non-null     int64  
 6   amrica_central                 73 non-null     int64  
 7   amrica_del_nord                73 non-null     int64  
 8   amrica_del_sud                 73 non-null     int64  
 9   austrlia_i_nova_zelanda        73 non-null     int64  
 10  carib                          73 non-null     int64  
 11  europa_meridional              73 non-null     int64  
 12  europa_occidental              73 non-null     int64

In [22]:
# Apilem primer per aplicar transformacions
poblacio = pd.concat([poblacio_23_raw, poblacio_15_raw], keys=[2023, 2015]).reset_index(names=["periode", "index"]).drop(columns=["index"])
poblacio.head()

,periode,codi_barri,poblacio_total,espanya,no_consta_x,resta_de_la_uni_europea,resta_del_mn,amrica_central,amrica_del_nord,amrica_del_sud,...,joves_espanya,joves_no_consta,joves_resta_de_la_uni_europea,joves_resta_del_mn,joves_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,2023,1,45671,22396,56,4563,18693,470,297,2888,...,3253,29,2362,11771,17415,0.509623,0.106413,0.490377,0.381314,0.241860
1,2023,2,24460,8385,32,3281,12784,453,306,2203,...,1435,16,1667,9196,12314,0.657195,0.146648,0.342805,0.503434,0.278250
2,2023,3,14274,8264,4,2619,3390,184,129,1248,...,1465,4,1369,2992,5830,0.421045,0.192518,0.578955,0.408435,0.305240
3,2023,4,22041,11764,16,4284,5989,313,390,1736,...,2016,16,2189,5049,9270,0.466267,0.212059,0.533733,0.420580,0.406288
4,2023,5,34403,23354,16,3445,7600,608,151,3088,...,3851,20,1270,6043,11184,0.321164,0.104526,0.678836,0.325088,0.377961


In [ ]:
# Calcul de percentatges
poblacio["pct_pob_estrangera"] = 1 -(poblacio["espanya"] / poblacio["poblacio_total"])
poblacio["pct_pob_estrangera_occidental"] = (poblacio["resta_de_la_uni_europea"] + poblacio["amrica_del_nord"]) / poblacio["poblacio_total"]
poblacio["pct_pob_espanyola"] = poblacio["espanya"] / poblacio["poblacio_total"]
poblacio["pct_joves"] = poblacio["joves_total"] / poblacio["poblacio_total"]
poblacio["pct_universitaris"] = (poblacio["estudis_superiors_espanya"] + poblacio["estudis_superiors_ue"] + poblacio["estudis_superiors_mon"]) / poblacio["poblacio_total"]

In [19]:
# Seleccionem conjunt d' atributs finals
poblacio = poblacio[["codi_barri", "periode", "poblacio_total", "pct_pob_estrangera", "pct_pob_estrangera_occidental", "pct_pob_espanyola", "pct_joves", "pct_universitaris"]]
poblacio.head()

,codi_barri,periode,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,1,2023,45671,0.509623,0.106413,0.490377,0.381314,0.241860
1,2,2023,24460,0.657195,0.146648,0.342805,0.503434,0.278250
2,3,2023,14274,0.421045,0.192518,0.578955,0.408435,0.305240
3,4,2023,22041,0.466267,0.212059,0.533733,0.420580,0.406288
4,5,2023,34403,0.321164,0.104526,0.678836,0.325088,0.377961


In [23]:
# Separem les dades per 2023 i 2015 per posteriorment crear un conjunt de modelling per als dos
poblacio_15 = poblacio[poblacio["periode"] == 2015].drop(columns= ["periode"])
poblacio_23 = poblacio[poblacio["periode"] == 2023].drop(columns= ["periode"])

In [24]:
poblacio_pivot = poblacio.pivot(index="codi_barri", columns="periode")
poblacio_pivot

poblacio_total        espanya        no_consta_x       \
periode              2015   2023    2015   2023        2015 2023   
codi_barri                                                         
1                   47150  45671   24498  22396          44   56   
2                   15514  24460    9007   8385          20   32   
3                   15037  14274   10331   8264           4    4   
4                   22468  22041   13644  11764          44   16   
5                   31548  34403   25417  23354          36   16   
...                   ...    ...     ...    ...         ...  ...   
69                  13374  13453   11131  10316           8    8   
70                  22844  27892   17336  17111           0   24   
71                  20163  21273   17654  17163           4   12   
72                  25971  26169   23267  21702           0   20   
73                  28727  29250   25768  24199           4    8   

           resta_de_la_uni_europea       resta_del_mn         ...  \
periode                       2015  2023         2015   2023  ...   
codi_barri                                                    ...   
1                             4347  4563        18293  18693  ...   
2                             3079  3281         3422  12784  ...   
3                             2178  2619         2527   3390  ...   
4                             4146  4284         4666   5989  ...   
5                             2340  3445         3778   7600  ...   
...                            ...   ...          ...    ...  ...   
69                            1088  1242         1153   1893  ...   
70                             592  1032         4916   9742  ...   
71                             754  1280         1756   2829  ...   
72                             598   897         2106   3565  ...   
73                             503   747         2458   4304  ...   

           pct_pob_estrangera           pct_pob_estrangera_occidental  \
periode                  2015      2023                          2015   
codi_barri                                                              
1                    0.480424  0.509623                      0.095949   
2                    0.419428  0.657195                      0.209037   
3                    0.312961  0.421045                      0.151160   
4                    0.392736  0.466267                      0.194632   
5                    0.194339  0.321164                      0.076455   
...                       ...       ...                           ...   
69                   0.167713  0.233182                      0.084492   
70                   0.241114  0.386527                      0.026440   
71                   0.124436  0.193203                      0.038734   
72                   0.104116  0.170698                      0.023372   
73                   0.103004  0.172684                      0.017823   

                     pct_pob_espanyola           pct_joves            \
periode         2023              2015      2023      2015      2023   
codi_barri                                                             
1           0.106413          0.519576  0.490377  0.394040  0.381314   
2           0.146648          0.580572  0.342805  0.428129  0.503434   
3           0.192518          0.687039  0.578955  0.373213  0.408435   
4           0.212059          0.607264  0.533733  0.403819  0.420580   
5           0.104526          0.805661  0.678836  0.296215  0.325088   
...              ...               ...       ...       ...       ...   
69          0.097822          0.832287  0.766818  0.269702  0.221661   
70          0.037681          0.758886  0.613473  0.287428  0.306898   
71          0.062239          0.875564  0.806797  0.280067  0.258591   
72          0.035309          0.895884  0.829302  0.235801  0.236960   
73          0.026120          0.896996  0.827316  0.236885  0.229641   

           pct_universitaris            
periode           

In [25]:
poblacio_pivot.columns = [f"{col}_{year}" for col, year in poblacio_pivot.columns]
poblacio_pivot = poblacio_pivot.reset_index()
poblacio_pivot.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,espanya_2015,espanya_2023,no_consta_x_2015,no_consta_x_2023,resta_de_la_uni_europea_2015,resta_de_la_uni_europea_2023,resta_del_mn_2015,...,pct_pob_estrangera_2015,pct_pob_estrangera_2023,pct_pob_estrangera_occidental_2015,pct_pob_estrangera_occidental_2023,pct_pob_espanyola_2015,pct_pob_espanyola_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023
0,1,47150,45671,24498,22396,44,56,4347,4563,18293,...,0.480424,0.509623,0.095949,0.106413,0.519576,0.490377,0.394040,0.381314,0.184963,0.241860
1,2,15514,24460,9007,8385,20,32,3079,3281,3422,...,0.419428,0.657195,0.209037,0.146648,0.580572,0.342805,0.428129,0.503434,0.333570,0.278250
2,3,15037,14274,10331,8264,4,4,2178,2619,2527,...,0.312961,0.421045,0.151160,0.192518,0.687039,0.578955,0.373213,0.408435,0.214737,0.305240
3,4,22468,22041,13644,11764,44,16,4146,4284,4666,...,0.392736,0.466267,0.194632,0.212059,0.607264,0.533733,0.403819,0.420580,0.323393,0.406288
4,5,31548,34403,25417,23354,36,16,2340,3445,3778,...,0.194339,0.321164,0.076455,0.104526,0.805661,0.678836,0.296215,0.325088,0.313966,0.377961


In [27]:
# Deltas
vars_delta = [
    "pct_pob_estrangera",
    "pct_pob_estrangera_occidental",
    "pct_joves",
    "pct_universitaris"
]

for var in vars_delta:
    poblacio_pivot[f"delta_{var}"] = poblacio_pivot[f"{var}_2023"] - poblacio_pivot[f"{var}_2015"]

poblacio_pivot["delta_pob_pct"] = (poblacio_pivot["poblacio_total_2023"] - poblacio_pivot["poblacio_total_2015"]
) / poblacio_pivot["poblacio_total_2015"]

poblacio_pivot.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,espanya_2015,espanya_2023,no_consta_x_2015,no_consta_x_2023,resta_de_la_uni_europea_2015,resta_de_la_uni_europea_2023,resta_del_mn_2015,...,pct_pob_espanyola_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_pob_pct
0,1,47150,45671,24498,22396,44,56,4347,4563,18293,...,0.490377,0.394040,0.381314,0.184963,0.241860,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,15514,24460,9007,8385,20,32,3079,3281,3422,...,0.342805,0.428129,0.503434,0.333570,0.278250,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,15037,14274,10331,8264,4,4,2178,2619,2527,...,0.578955,0.373213,0.408435,0.214737,0.305240,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,22468,22041,13644,11764,44,16,4146,4284,4666,...,0.533733,0.403819,0.420580,0.323393,0.406288,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,31548,34403,25417,23354,36,16,2340,3445,3778,...,0.678836,0.296215,0.325088,0.313966,0.377961,0.126825,0.028071,0.028873,0.063995,0.090497


In [30]:
# Ens quedem amb els deltas per al dataset final de poblacio
poblacio_final = poblacio_pivot[[col for col in poblacio_pivot.columns if col.startswith("delta") or col == "codi_barri"]]
poblacio_final.head()

,codi_barri,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_pob_pct
0,1,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,0.126825,0.028071,0.028873,0.063995,0.090497


# Dades Econòmiques
Incorporem dades econòmiques:
- Renda neta mitja per persona.
- Índex de gini

In [36]:
econs_15 = pd.read_csv("../data/processed/df_economiques_15.csv")
econs_15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codi_barri    73 non-null     int64  
 1   import_euros  73 non-null     float64
 2   index_gini    73 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 1.8 KB


In [37]:
econs_23 = pd.read_csv("../data/processed/df_economiques_23.csv")
econs_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codi_barri    73 non-null     int64  
 1   import_euros  73 non-null     float64
 2   index_gini    73 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 1.8 KB


In [45]:
# Combinem
econs = dim_barris[["codi_barri"]].copy()
econs_merged = econs.merge(econs_23, on="codi_barri", how="left")\
    .merge(econs_15, on = "codi_barri", suffixes=["_2023", "_2015"])

econs_merged.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015
0,1,11917.476190,33.795238,8507.809524,36.338095
1,2,16013.222222,39.266667,11447.444444,41.087500
2,3,15359.818182,32.809091,10818.818182,34.954545
3,4,16621.153846,36.769231,11689.000000,39.515385
4,5,20418.800000,31.890000,15441.500000,34.020000


In [47]:
# Calculem deltes
econs_final = econs_merged.copy()
econs_final["delta_renda_pct"] = (econs_final["import_euros_2023"] - econs_final["import_euros_2015"]) / econs_final["import_euros_2015"]
econs_final["delta_gini_pct"] = (econs_final["index_gini_2023"] - econs_final["index_gini_2015"]) / econs_final["index_gini_2015"]

econs_final.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015,delta_renda_pct,delta_gini_pct
0,1,11917.476190,33.795238,8507.809524,36.338095,0.400769,-0.069978
1,2,16013.222222,39.266667,11447.444444,41.087500,0.398847,-0.044316
2,3,15359.818182,32.809091,10818.818182,34.954545,0.419732,-0.061378
3,4,16621.153846,36.769231,11689.000000,39.515385,0.421948,-0.069496
4,5,20418.800000,31.890000,15441.500000,34.020000,0.322333,-0.062610


In [48]:
# Ens  quedem amb els deltes només
econs_final = econs_final[[col for col in econs_final.columns if col.startswith("delta") or col == "codi_barri"]]
econs_final.head()

,codi_barri,delta_renda_pct,delta_gini_pct
0,1,0.400769,-0.069978
1,2,0.398847,-0.044316
2,3,0.419732,-0.061378
3,4,0.421948,-0.069496
4,5,0.322333,-0.062610


# Dades Urbanes

# Dades habitatge